# Basic RAG

Basic RAG with Langchain

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 09/03/2026   | Martin | Created   | Started basic RAG application | 
| 11/03/2026   | Martin | Updated   | Completed the creation of a basic RAG | 

# Content

* [Introduction](#introduction)

# Introduction

This demonstrates the fundamental RAG pattern:
  1. Load documents
  2. Split into chunks
  3. Embed and store in ChromaDB
  4. Retrieve relevant chunks on query
  5. Generate answer with LLM

ARCHITECTURAL DECISIONS:
- RecursiveCharacterTextSplitter: Tries to split on natural boundaries
  (paragraphs → sentences → words) before resorting to character splits.
  This preserves semantic coherence within chunks.

- chunk_size=500, chunk_overlap=50: A starting heuristic. Too small = 
  missing context; too large = noisy retrieval + hitting context limits.
  Overlap ensures sentences split across boundaries aren't lost.

- HuggingFaceEmbeddings (all-MiniLM-L6-v2): A lightweight, fast model
  that runs locally without API costs. For production, consider 
  text-embedding-ada-002 (OpenAI) or text-embedding-3-small for better
  quality at scale.

- ChromaDB (in-memory): Zero-setup vector store, ideal for development.
  In production you'd persist to disk or use a hosted solution.

- k=3 retrieval: Fetching top-3 chunks balances context richness vs.
  noise. Too many chunks overwhelm the LLM; too few miss relevant info.

In [1]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [2]:
# ==============================
# Document Loading
# ==============================
def load_documents(data_dir: str = "./data"):
  """
  Loads text files from a directory. Extend this with different loaders for
  PDFs, webpages, databases, and other knowledge sources
  """
  loader = DirectoryLoader(
    path=data_dir,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True
  )
  docs = loader.load()
  print(f"[LOAD] Loaded {len(docs)} documents from '{data_dir}'")

  return docs

In [3]:
docs = load_documents()

100%|██████████| 3/3 [00:00<00:00, 770.21it/s]

[LOAD] Loaded 3 documents from './data'


Chunking helps to retrieve only the relevant pieces from a document. Wrong chunk size = wrong answers.

In [4]:
# ==============================
# Chunking
# ==============================
def chunk_documents(docs, chunk: int = 500, chunk_overlap: int = 50):
  """
  RecursiveCharacterTextSplitter priority order:
  ["\\n\\n", "\\n", " ", ""] — paragraph → line → word → char
  """
  splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk,
    chunk_overlap=chunk_overlap,
    length_function=len,
    add_start_index=True # Stores char offset in metadata: useful for highlighting source passage
  )
  chunks = splitter.split_documents(docs)
  print(f"[CHUNK] Split into {len(chunks)} chunks (size={chunk}, overlap={chunk_overlap})")

  return chunks

In [5]:
chunks = chunk_documents(docs, chunk=500, chunk_overlap=50)
chunks

[CHUNK] Split into 21 chunks (size=500, overlap=50)


[Document(metadata={'source': 'data/ml_fundamentals.txt', 'start_index': 0}, page_content='Machine Learning Fundamentals\n\nSupervised Learning\nSupervised learning is a type of machine learning where the model is trained on labeled data. The algorithm learns to map input features to output labels by minimizing a loss function. Common examples include linear regression for continuous outputs and logistic regression for classification tasks.'),
 Document(metadata={'source': 'data/ml_fundamentals.txt', 'start_index': 352}, page_content='Key supervised learning algorithms include:\n- Decision Trees: Tree-like models that split data based on feature thresholds\n- Random Forests: Ensemble of decision trees that reduces overfitting\n- Support Vector Machines (SVM): Finds optimal hyperplane to separate classes\n- Neural Networks: Layered architectures inspired by biological neurons\n- Gradient Boosting (XGBoost, LightGBM): Sequential ensemble methods'),
 Document(metadata={'source': 'data/ml_

Using embeddings are better than keyword searches because it allows the similarity search to be performed semantically, especially good for conceptual questions

In [6]:
# ==============================
# Embedding + Vector Store
# ==============================
def build_vectorstore(chunks, persist_dir: str = None):
  """
  Embeddings are an expensive step. Pre-compute and persist in vectorstore
  in Production

  all-MiniLM-L6-v2 is a sentence transformer
  """
  print("[EMBED] Loading embedding model...")
  embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
  )

  print("[STORE] Building ChromaDB vectorstore...")
  kwargs = {"documents": chunks, "embedding": embeddings}
  if persist_dir:
    kwargs["persist_directory"] = persist_dir
  store = Chroma.from_documents(**kwargs)
  print(f"[STORE] Stored {store._collection.count()} vectors in ChromaDB")
  return store, embeddings


In [7]:
store = build_vectorstore(chunks)
store

[EMBED] Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[STORE] Building ChromaDB vectorstore...
[STORE] Stored 21 vectors in ChromaDB


(<langchain_community.vectorstores.chroma.Chroma at 0x13ddb4150>,
 HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={'device': 'cpu'}, encode_kwargs={'normalize_embeddings': True}, query_encode_kwargs={}, multi_process=False, show_progress=False))

Retrieval converts a query into an embedding and finds the k most similar chunks via cosine similarity (or wtv similarity metric chosen)

In [12]:
# ==============================
# Retrieval
# ==============================
def build_retriever(store, k: int = 3):
  """
  k=3 is a reasonable default. Production depends on:
  - Average answer complexity
  - LLM context window
  - Latency requirements
  """
  retriever = store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": k}
  )

  return retriever

In [15]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
  ("system", (
    "You are a helpful assistant. Use ONLY the following context to answer "
    "the question. If the answer isn't in the context, say \"I don't know "
    "based on the provided documents.\"\n\n"
    "Context:\n{context}"
  )),
  ("human", "{question}"),
])

HOW THE PIPE WORKS: The | operator connects Runnables into a directed pipeline. Each component
receives the output of the previous one as its input.

`RunnableParallel` runs its branches concurrently and merges results into a dict:
  - "context": invokes the retriever, then formats the Document list into a string
  - "question": RunnablePassthrough() forwards the original query unchanged

In [27]:
# ==============================
# QA Chain
# ==============================
def _format_docs(docs) -> str:
  """Concatenate docus chunks into 1 long string"""
  return "\n\n".join(doc.page_content for doc in docs)

def qa_chain(retriever, llm):
  """Uses a RunnableParallel and the LCEL syntax to create a chain that
  feeds the output of each component to the next"""
  chain = (
    RunnableParallel({
      "context": retriever | _format_docs,
      "question": RunnablePassthrough()
    })
    | RAG_PROMPT
    | llm
    | StrOutputParser()
  )

  return chain

def build_qa_chain_with_sources(retriever, llm):
  """
  Variant that preserves retrieved Document objects in the output so callers
  can render source citations — equivalent to return_source_documents=True.

  RunnableParallel is used twice:
    1. First pass: retrieve docs and pass the question through in parallel.
    2. Second pass: generate the answer AND carry the raw docs forward.

  Output dict: {"question": str, "context": List[Document], "answer": str}
  """
  # 1. Retreive + Pass through questions
  retrieve = RunnableParallel({
    "context": retriever,
    "question": RunnablePassthrough()
  })

  # 2. Generate answers + provide source
  generate = RunnableParallel({
    "answer": (
      RunnableParallel({
        "context": lambda x: _format_docs(x['context']),
        "question": lambda x: x['question']
      })
      | RAG_PROMPT
      | llm
      | StrOutputParser()
    ),
    "context": lambda x: x['context'],
    "question": lambda x: x['question']
  })

  return retrieve | generate

Model: Qwen3 0.6B via Ollama 

In [28]:
def rag_demo():
  print("\n" + "="*60)
  print("  BASIC RAG PIPELINE DEMO")
  print("="*60)

  # Initialise the LLM
  from langchain_ollama import OllamaLLM
  llm = OllamaLLM(
    model="qwen3:0.6b",
    temperature=0
  )

  # Build the pipeline
  data_dir = "./data"
  docs = load_documents(data_dir)
  chunks = chunk_documents(docs)
  vectorstore, _ = build_vectorstore(chunks)
  retriever = build_retriever(vectorstore, k=3)

  # QA chain using the sources variant
  qa_chain = build_qa_chain_with_sources(retriever, llm)

  # Run some queries
  queries = [
    "What is supervised learning and what algorithms does it use?",
    "How do you prevent overfitting in machine learning models?",
    "How does Retrieval Augmented Generation work?",
    "What is the best package to implement RAGs?"
  ]

  print("\n--- QUERY RESULTS ---\n")
  for query in queries:
    print(f"Q: {query}")
    result = qa_chain.invoke(query)
    print(f"A: {result['answer']}")

    sources = {doc.metadata.get("source", "unknown") for doc in result['context']}
    print(f"    Sources: {', '.join(os.path.basename(s) for s in sources)}")
    print()
  
  return vectorstore

In [31]:
rag_demo()


  BASIC RAG PIPELINE DEMO


100%|██████████| 3/3 [00:00<00:00, 3223.08it/s]

[LOAD] Loaded 3 documents from './data'
[CHUNK] Split into 21 chunks (size=500, overlap=50)
[EMBED] Loading embedding model...



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13328.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[STORE] Building ChromaDB vectorstore...
[STORE] Stored 105 vectors in ChromaDB

--- QUERY RESULTS ---

Q: What is supervised learning and what algorithms does it use?
A: Supervised learning is a type of machine learning where the model learns to map input features to output labels by minimizing a loss function. It uses algorithms like linear regression and logistic regression.
    Sources: ml_fundamentals.txt

Q: How do you prevent overfitting in machine learning models?
A: To prevent overfitting, you can use techniques like L1 (Lasso), L2 (Ridge), dropout, early stopping, and cross-validation.
    Sources: ml_fundamentals.txt

Q: How does Retrieval Augmented Generation work?
A: Retrieval Augmented Generation (RAG) works by first retrieving relevant documents from a knowledge base using a retrieval system (e.g., vector similarity search), then using that information to generate an answer conditioned on the retrieved context. This approach ensures that responses are grounded in factual

In [ ]:
%load_ext watermark
%watermark